In [1]:
import numpy as np
import pandas as pd
import illustris_python as il
from scipy.optimize import minimize_scalar
from scipy.stats import binned_statistic_2d
import os 

# --- datos ---
basePath = '/home/tnguser/sims.TNG/TNG100-1/output/'
snapshot = 99
output_folder = "gas2_sn99"

# tipos de partícula
GAS = 0
STAR = 4

# Lista de SubhaloIDs a procesar
subhalo_ids = [
    41, 17211, 52630, 60744, 76093, 108013, 114398, 131050, 146181, 146184, 
    149174, 168392, 181987, 188861, 192904, 212838, 254112, 255616, 279708, 279709, 
    306517, 325337, 328678, 329850, 344593, 360510, 361190, 362552, 367929, 368884, 
    369366, 376739, 382484, 382922, 385350, 386429, 388819, 393337, 395154, 395736, 
    400547, 405746, 406256, 407321, 409226, 409358, 419061, 420304, 420435, 424255, 
    426361, 429127, 430070, 430183, 430470, 434410, 435417, 437314, 437822, 438297, 
    438787, 441327, 441397, 441857, 445567, 446549, 447433, 449972, 450046, 451410, 
    453081, 453313, 453961, 455335, 456725, 459850, 461283, 461609, 461667, 461806, 
    461864, 462481, 462775, 464742, 466387, 467212, 467798, 467871, 468590, 469315, 
    470322, 470380, 470617, 471665, 472457, 473004, 473471, 473514, 473898, 474813, 
    474979, 475020, 475656, 475906, 476487, 476997, 477089, 478696, 480194, 484771, 
    485283, 485518, 487244, 488123, 488415, 489100, 489593, 492425, 492825, 493491, 
    493964, 494771, 495442, 497646, 498768, 499130, 499609, 501215, 503503, 503664, 
    504650, 504837, 507965, 509103, 509468, 511195, 511858, 512154, 516398, 517366, 
    520839, 521153
]
print(f"Total de subhalos cargados: {len(subhalo_ids)}")

# Crear la carpeta de salida 
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Carpeta '{output_folder}' creada.")

# --- bucle ---
for subhalo_id in subhalo_ids:
    print(f"\nProcesando Subhalo ID: {subhalo_id} (GAS)...")
    
    try:
        # Cargar propiedades del subhalo
        subhalo = il.groupcat.loadSingle(basePath, snapshot, subhaloID=subhalo_id)
        subhalo_pos = subhalo['SubhaloPos']
        x0, y0 = subhalo_pos[0], subhalo_pos[1]
        v_sys = subhalo['SubhaloVel']
        
        # CRUCIAL: Usamos el radio medio ESTELAR [4] como escala de referencia pero luego usamos
        # el radio medio del gas, por eso hay 2 graficas para las curvas de rotación. 
        rhalf_stars = subhalo['SubhaloHalfmassRadType'][GAS]

        # Cargar partículas de GAS (partType=0)
        parts = il.snapshot.loadSubhalo(
            basePath, snapshot, subhalo_id,
            partType=GAS,
            fields=['Coordinates', 'Velocities', 'Masses']
        )

        # Verificación  contra KeyError / Subhalos sin gas
        if parts is None or 'Coordinates' not in parts or len(parts['Coordinates']) == 0:
            print(f"Advertencia: El subhalo {subhalo_id} no tiene partículas de gas. Saltando...")
            continue

        coords = parts['Coordinates']
        masses = parts['Masses'] * 1e10

        # Centrar y filtrar por radio (3 * rhalf estelar o gas) IMPORTANTE: dejamos en gas
        #Aunque dejemos como rhalf_stars es del gas
        x_rel = coords[:, 0] - x0
        y_rel = coords[:, 1] - y0
        r_proj = np.sqrt(x_rel**2 + y_rel**2)
        mask_r = r_proj < 3 * rhalf_stars
        
        if np.count_nonzero(mask_r) < 50:
            print(f"Advertencia: Menos de 50 partículas de gas dentro de 3*rhalf para {subhalo_id}. Saltando...")
            continue

        x_f = x_rel[mask_r]
        y_f = y_rel[mask_r]
        m_f = masses[mask_r]

        # Tensor de inercia e Inclinación del GAS
        I_xx = np.sum(m_f * x_f**2)
        I_yy = np.sum(m_f * y_f**2)
        I_xy = np.sum(m_f * x_f * y_f)
        I_tensor = np.array([[I_xx, I_xy], [I_xy, I_yy]])

        eigvals, eigvecs = np.linalg.eigh(I_tensor)
        a_semi, b_semi = np.sqrt(eigvals[1] + 1e-30), np.sqrt(eigvals[0] + 1e-30)
        i_deg = np.degrees(np.arccos(np.clip(b_semi / a_semi, 0, 1)))

        # generacion de la malla (Extensión espacial del mapa)
        nbins = 85
        size_total = 6.0 * rhalf_stars  # Radio de 3*rhalf hacia cada lado
        dd = size_total / 2.0
        extent = [x0 - dd, x0 + dd, y0 - dd, y0 + dd]

        x_abs = coords[:, 0]
        y_abs = coords[:, 1]
        vz = coords_vlos = parts['Velocities'][:, 2] - v_sys[2] # Velocidad en LOS (Z)

        # MAPA DE VELOCIDAD VECTORIZADO (Rápido y eficiente)
        # 1. Calcular la mediana por celda
        v_median, xedges, yedges, _ = binned_statistic_2d(
            x_abs, y_abs, vz, 
            statistic='median', 
            bins=nbins, 
            range=[[extent[0], extent[1]], [extent[2], extent[3]]]
        )
        
        # 2. Calcular el conteo por celda para aplicar el filtro de min_particles
        v_count, _, _, _ = binned_statistic_2d(
            x_abs, y_abs, vz, 
            statistic='count', 
            bins=nbins, 
            range=[[extent[0], extent[1]], [extent[2], extent[3]]]
        )

        # Construir Vmap final transponiendo para el formato correcto (y, x)
        Vmap = v_median.T
        count_map = v_count.T
        Vmap[count_map < 9] = np.nan  # Filtro de mínimo 9 partículas por píxel

        # Extraer píxeles válidos para el gradiente cinemático
        yy, xx = np.meshgrid((yedges[:-1] + yedges[1:]) / 2, (xedges[:-1] + xedges[1:]) / 2)
        #mapa tiene píxeles vacíos (NaN) debido al filtro de un mínimo de 9 partículas
        valid_mask = ~np.isnan(Vmap)
        
        if not np.any(valid_mask):
            print(f"Advertencia: No hay suficientes píxeles válidos para {subhalo_id}. Saltando...")
            continue

        X_valid = xx[valid_mask] - x0
        Y_valid = yy[valid_mask] - y0
        V_valid = Vmap[valid_mask]

        # Maximizar gradiente cinemático
        def velocity_gradient_score(theta_deg):
            theta = np.radians(theta_deg)
            axis = np.cos(theta) * X_valid + np.sin(theta) * Y_valid
            # Evitar errores si la desviación estándar es cero
            if np.std(axis) == 0 or np.std(V_valid) == 0:
                return 0
            #coeficiente de correlación de Pearson (‭$r$‬) entre la posición proyectada en esa línea (axis) y la velocidad real del píxel (V_valid).
            return -np.abs(np.corrcoef(axis, V_valid)[0, 1])

        result = minimize_scalar(velocity_gradient_score, bounds=(0, 180), method='bounded')
        phi0_kin_deg = result.x

        # EXPORTACIÓN
        df_export = pd.DataFrame({
            "r_kpc": np.sqrt(X_valid**2 + Y_valid**2),
            "phi_rad": np.arctan2(Y_valid, X_valid),
            "Vobs_kms": V_valid
        })

        file_name = f"{phi0_kin_deg:.2f}_{i_deg:.2f}_shid{subhalo_id}_gas.csv"
        file_path = os.path.join(output_folder, file_name)
        df_export.to_csv(file_path, index=False)

        print(f"-> Guardado exitoso: {file_name}")

    except Exception as e:
        print(f"Error procesando subhalo {subhalo_id}: {e}")

print("\n--- Proceso finalizado ---")

Total de subhalos cargados: 142
Carpeta 'gas2_sn99' creada.

Procesando Subhalo ID: 41 (GAS)...
-> Guardado exitoso: 56.59_37.11_shid41_gas.csv

Procesando Subhalo ID: 17211 (GAS)...
-> Guardado exitoso: 180.00_66.35_shid17211_gas.csv

Procesando Subhalo ID: 52630 (GAS)...
-> Guardado exitoso: 128.94_41.78_shid52630_gas.csv

Procesando Subhalo ID: 60744 (GAS)...
-> Guardado exitoso: 12.62_39.05_shid60744_gas.csv

Procesando Subhalo ID: 76093 (GAS)...
-> Guardado exitoso: 103.65_67.18_shid76093_gas.csv

Procesando Subhalo ID: 108013 (GAS)...
-> Guardado exitoso: 12.71_71.16_shid108013_gas.csv

Procesando Subhalo ID: 114398 (GAS)...
-> Guardado exitoso: 117.16_68.26_shid114398_gas.csv

Procesando Subhalo ID: 131050 (GAS)...
-> Guardado exitoso: 33.89_58.21_shid131050_gas.csv

Procesando Subhalo ID: 146181 (GAS)...
-> Guardado exitoso: 126.58_21.61_shid146181_gas.csv

Procesando Subhalo ID: 146184 (GAS)...
-> Guardado exitoso: 63.79_63.07_shid146184_gas.csv

Procesando Subhalo ID: 149174 